# ProtGPT2 scoring for decoding-design-bias

GPT-2-style decoder-only protein language model (Ferruz et al., 2022).

**Critical difference from every other model in this pipeline:** ProtGPT2
tokenises protein sequences as **BPE chunks**, not per residue. A
100-AA protein tokenises to roughly 25–35 BPE tokens, depending on the
sequence. Every other scoring script in this directory tokenises one
token per residue, so the BPE step needs explicit handling at
normalisation time.

Sequence-only — uses the `sequence` field of the dataset CSV (the same
input as ESM2). No PDB / structure dependency.

Scoring convention:
- For each WT sequence `s = s_1 ... s_L`, BPE-tokenise (no special tokens
  added by default; we prepend GPT-2's `<|endoftext|>` as BOS so position
  0's prediction has context).
- Single teacher-forced forward pass; shifted-logits trick to compute
  `sum_t log P(token_t | tokens_{<t})` over the *T* BPE tokens.
- **Normalise by residue count `L`, not by token count `T`.** This is
  the user-confirmed choice: every other script in the pipeline reports
  per-residue means, so `protgpt2_score = total_logp_over_BPE_tokens / L`
  keeps ProtGPT2 on the same y-axis as ProteinMPNN / ESM-IF / ESM2 /
  Caliby / TriFlow / ESM3 in the variance decomposition.
- For transparency the output also exposes `protgpt2_total_logp`
  (the BPE sum) and `num_tokens` (T), so a reader can reconstruct the
  per-token mean as `total_logp / num_tokens` if desired.

Sequence-cleaning policy (matches ESM2 + ProGen2):
- 20 standard amino acids only.
- Terminal `*` stripped if present; no replacement of X / B / Z / U / O.
- Proteins containing non-standard residues are recorded with NaN
  scores and `sequence_filter_status="nonstandard_amino_acid"`.

Checkpoint: `nferruz/ProtGPT2` (only public ProtGPT2 release).

## 1. Setup

In [ ]:
!nvidia-smi -L


In [ ]:
# transformers ships GPT-2 + GPT2Tokenizer; no architecture upload needed
# (unlike ProGen2 which uses a custom config + trust_remote_code).
!pip install -q 'transformers>=4.40,<5' 'tokenizers>=0.15' 'accelerate>=0.30' biopython tqdm


In [ ]:
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/LBDillon/decoding-design-bias.git'
REPO_DIR = '/content/decoding-design-bias'
DATASET  = '/content/main_plus_r2_r3_scored_filterC_v4.csv'  # edit if needed

DRIVE_OUT_DIR = '/content/drive/MyDrive/decoding-design-bias/outputs'
OUTPUT        = f'{DRIVE_OUT_DIR}/protgpt2_scores.csv'
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

PROTGPT2_CHECKPOINT = 'nferruz/ProtGPT2'
MAX_TOKENS_PER_FORWARD = 2048  # halved on OOM

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=False)

assert os.path.exists(DATASET), DATASET
print('output ->', OUTPUT)
print('checkpoint ->', PROTGPT2_CHECKPOINT)


## 2. Load model

In [ ]:
import torch
import warnings; warnings.filterwarnings('ignore')
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

assert torch.cuda.is_available(), 'Use a GPU runtime (Runtime -> Change runtime type -> GPU).'
device = torch.device('cuda')

tokenizer = GPT2TokenizerFast.from_pretrained(PROTGPT2_CHECKPOINT)
model = GPT2LMHeadModel.from_pretrained(
    PROTGPT2_CHECKPOINT,
    torch_dtype=torch.bfloat16,
).to(device).eval()

# ProtGPT2 inherits GPT-2's tokeniser, so BOS is <|endoftext|>.
# Prepend this so position 0's prediction is conditioned on something rather
# than being a free-standing first-token guess.
BOS_ID = tokenizer.bos_token_id
if BOS_ID is None:
    BOS_ID = tokenizer.eos_token_id  # GPT-2 reuses EOS as BOS
print(f'BOS_ID: {BOS_ID}  (token: {tokenizer.decode([BOS_ID]) if BOS_ID is not None else None})')
print(f'vocab size: {tokenizer.vocab_size}')


## 3. Sequence cleaning + scoring wrapper

In [ ]:
import pandas as pd
import math

VALID_STANDARD_AA = set('ACDEFGHIKLMNPQRSTVWY')

def clean_sequence(seq):
    """Match the ESM2 / ProGen2 cleaning policy."""
    if seq is None or (isinstance(seq, float) and math.isnan(seq)):
        return '', 'missing'
    s = str(seq).strip().upper().replace(' ', '').replace('\n', '').replace('\r', '')
    if s.endswith('*'):
        s = s[:-1]
    bad_chars = ''.join(sorted(set(s) - VALID_STANDARD_AA))
    return s, bad_chars


def classify(seq_clean, bad_chars):
    if not seq_clean:
        return 'empty_sequence'
    if bad_chars:
        return 'nonstandard_amino_acid'
    return 'included'


@torch.no_grad()
def score_sequence_protgpt2(seq):
    """Teacher-forced log-likelihood for one WT sequence under ProtGPT2.

    BPE-tokenise (no specials added), prepend BOS, single forward pass,
    shifted-logits to compute next-token log-probs.

    Returns
      - protgpt2_total_logp : sum log p over BPE tokens (== full sequence LL)
      - protgpt2_score      : per-RESIDUE mean log p (= total / L, NOT / T)
      - protgpt2_per_token  : per-TOKEN mean log p (= total / T), exposed
                               for transparency / methods-section comparison
      - scored_length       : L (number of residues)
      - num_tokens          : T (number of BPE tokens scored, after BOS)
    """
    L = len(seq)
    enc = tokenizer(seq, add_special_tokens=False, return_tensors='pt')
    token_ids = enc['input_ids'][0]
    # Prepend BOS so position 0 has context
    if BOS_ID is not None:
        token_ids = torch.cat([torch.tensor([BOS_ID]), token_ids])
    token_ids = token_ids.to(device)

    out = model(input_ids=token_ids.unsqueeze(0))
    logits = out.logits[0]                  # (T+1, V)
    log_probs = torch.log_softmax(logits.float(), dim=-1)

    next_tokens   = token_ids[1:]           # (T,)
    pred_logprobs = log_probs[:-1]          # (T, V)
    picked = pred_logprobs[torch.arange(next_tokens.shape[0], device=device), next_tokens]

    total_logp = float(picked.sum().item())
    T = int(next_tokens.shape[0])
    return {
        'protgpt2_total_logp': total_logp,
        'protgpt2_score':      total_logp / L if L else float('nan'),
        'protgpt2_per_token':  total_logp / T if T else float('nan'),
        'scored_length':       L,
        'num_tokens':          T,
    }


def score_with_oom_retry(seq):
    """Catch OOM / general errors so a single failure doesn't kill the run."""
    try:
        return score_sequence_protgpt2(seq), None
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return None, f'out_of_memory: L={len(seq)}'
    except Exception as exc:
        return None, repr(exc)


## 4. Sanity check on 3 proteins

In [ ]:
import csv, time

with open(DATASET) as fh:
    rows = list(csv.DictReader(fh))
print('dataset:', len(rows), 'rows')

for r in rows[:3]:
    entry = r['Entry']
    seq, bad = clean_sequence(r.get('sequence', ''))
    status = classify(seq, bad)
    if status != 'included':
        print(entry, status, bad); continue
    t0 = time.time(); res, err = score_with_oom_retry(seq); t = time.time() - t0
    if res is None:
        print(entry, 'failed:', err); continue
    print(f'{entry} L={res["scored_length"]:4d} T={res["num_tokens"]:4d} '
          f'score(per-res)={res["protgpt2_score"]:.4f}  '
          f'per_token={res["protgpt2_per_token"]:.4f}  '
          f'total={res["protgpt2_total_logp"]:.2f}  ({t:.1f}s)')


## 5. Full run with resume

In [ ]:
from tqdm.auto import tqdm

already = set()
if os.path.exists(OUTPUT):
    with open(OUTPUT) as fh:
        for row in csv.DictReader(fh):
            if row.get('Entry'):
                already.add(row['Entry'])
    print('resuming, already scored:', len(already))

todo = [r for r in rows if r['Entry'] not in already]
todo.sort(key=lambda r: len(r.get('sequence', '')))
print('remaining:', len(todo))

open_mode = 'a' if already else 'w'
FIELDS = [
    'Entry', 'species', 'domain',
    'protgpt2_score', 'protgpt2_total_logp', 'protgpt2_per_token',
    'scored_length', 'num_tokens',
    'dataset_length', 'sequence_filter_status', 'error',
]

with open(OUTPUT, open_mode, newline='') as out:
    w = csv.DictWriter(out, fieldnames=FIELDS)
    if open_mode == 'w':
        w.writeheader()

    counts = {'ok': 0, 'skipped_nonstd': 0, 'oom': 0, 'error': 0, 'empty': 0}
    t_start = time.time()
    for r in tqdm(todo, desc='protgpt2'):
        entry = r['Entry']
        rec = {
            'Entry': entry,
            'species': r.get('species', ''),
            'domain':  r.get('domain', ''),
            'protgpt2_score': '',
            'protgpt2_total_logp': '',
            'protgpt2_per_token': '',
            'scored_length': 0,
            'num_tokens': 0,
            'dataset_length': len(r.get('sequence', '')),
            'sequence_filter_status': '',
            'error': '',
        }
        seq, bad = clean_sequence(r.get('sequence', ''))
        status = classify(seq, bad)
        rec['sequence_filter_status'] = status
        if status != 'included':
            if status == 'empty_sequence':
                counts['empty'] += 1
            else:
                counts['skipped_nonstd'] += 1
                rec['error'] = f'bad_chars={bad}'
            w.writerow(rec); out.flush(); continue

        res, err = score_with_oom_retry(seq)
        if res is None:
            rec['error'] = err or 'unknown_error'
            if err and 'out_of_memory' in err:
                counts['oom'] += 1
            else:
                counts['error'] += 1
            w.writerow(rec); out.flush(); continue

        rec.update(res)
        w.writerow(rec); out.flush()
        counts['ok'] += 1
        if counts['ok'] % 50 == 0:
            torch.cuda.empty_cache()

print('done in', round(time.time() - t_start, 1), 's', counts)


## 6. Quick look

In [ ]:
import pandas as pd
df = pd.read_csv(OUTPUT)
print(df.shape)
print('non-null protgpt2_score:', df['protgpt2_score'].notna().sum())
print('mean BPE tokens / residue:',
      (df['num_tokens'] / df['scored_length']).dropna().mean())
df[['protgpt2_score', 'protgpt2_per_token', 'protgpt2_total_logp',
    'scored_length', 'num_tokens']].describe()
